# COVID-19 in India: Data Cleaning

Cleans the state-wise COVID-19 dataset (30 Jan 2020 to 11 Aug 2021) so it is ready for analysis.

**Input:** `data/raw.csv` → **Output:** `data/clean.csv`

**Steps:** 1. Load → 2. Inspect → 3. Fix each problem → 4. Final check and save

## 1. Import Libraries and Load the Data

In [4]:
import pandas as pd
from pathlib import Path

In [5]:
csv_path = Path("../data/covid_19_india.csv")
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

(18110, 9)


,Sno,Date,Time,State/UnionTerritory,ConfirmedIndianNational,ConfirmedForeignNational,Cured,Deaths,Confirmed
0,1,2020-01-30,6:00 PM,Kerala,1,0,0,0,1
1,2,2020-01-31,6:00 PM,Kerala,1,0,0,0,1
2,3,2020-02-01,6:00 PM,Kerala,2,0,0,0,2
3,4,2020-02-02,6:00 PM,Kerala,3,0,0,0,3
4,5,2020-02-03,6:00 PM,Kerala,3,0,0,0,3


## 2. Inspect the Data

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18110 entries, 0 to 18109
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Sno                       18110 non-null  int64
 1   Date                      18110 non-null  str  
 2   Time                      18110 non-null  str  
 3   State/UnionTerritory      18110 non-null  str  
 4   ConfirmedIndianNational   18110 non-null  str  
 5   ConfirmedForeignNational  18110 non-null  str  
 6   Cured                     18110 non-null  int64
 7   Deaths                    18110 non-null  int64
 8   Confirmed                 18110 non-null  int64
dtypes: int64(4), str(5)
memory usage: 1.2 MB


In [7]:
print("Nulls per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Nulls per column:
Sno                         0
Date                        0
Time                        0
State/UnionTerritory        0
ConfirmedIndianNational     0
ConfirmedForeignNational    0
Cured                       0
Deaths                      0
Confirmed                   0
dtype: int64

Duplicate rows: 0


**Inspection findings**
- 18,110 rows and 9 columns, with no nulls and no duplicate rows.
- That looks clean, but `isna()` only finds real nulls, so hidden problems can still exist.
- `Date` is stored as text (`str`) instead of a date.

## 3. Problems and Fixes

Each problem follows the same steps: **look → check → fix → note**.

### Problem 1: `Date` is text, and two columns are not useful
`Date` is stored as text, so it cannot be sorted or plotted as a date. `Sno` is only a row number and `Time` adds no useful information, so both are dropped.

In [8]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.drop(columns=["Sno", "Time"])

print(df["Date"].dtype)
print(df.shape)

datetime64[us]
(18110, 7)


### Problem 2: Typos and star marks in state names
One state appears under several names. Look at all the names, sorted, so the odd ones stand out.

In [9]:
sorted(df["State/UnionTerritory"].unique())

['Andaman and Nicobar Islands',
 'Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Bihar****',
 'Cases being reassigned to states',
 'Chandigarh',
 'Chhattisgarh',
 'Dadra and Nagar Haveli',
 'Dadra and Nagar Haveli and Daman and Diu',
 'Daman & Diu',
 'Delhi',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Himanchal Pradesh',
 'Jammu and Kashmir',
 'Jharkhand',
 'Karanataka',
 'Karnataka',
 'Kerala',
 'Ladakh',
 'Lakshadweep',
 'Madhya Pradesh',
 'Madhya Pradesh***',
 'Maharashtra',
 'Maharashtra***',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Puducherry',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Telengana',
 'Tripura',
 'Unassigned',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal']

The names `Maharashtra***`, `Bihar****` and `Madhya Pradesh***` carry star marks, and `Karanataka`, `Telengana` and `Himanchal Pradesh` are misspelled. Map each wrong name to the correct one.

In [10]:
state_fix = {
    "Maharashtra***": "Maharashtra",
    "Himanchal Pradesh": "Himachal Pradesh",
    "Bihar****": "Bihar",
    "Telengana": "Telangana",
    "Karanataka": "Karnataka",
    "Madhya Pradesh***": "Madhya Pradesh",
}

print("Names before:", df["State/UnionTerritory"].nunique())
df["State/UnionTerritory"] = df["State/UnionTerritory"].replace(state_fix)
print("Names after: ", df["State/UnionTerritory"].nunique())

# Check: renaming must not create two rows for the same state on the same date
print("Duplicate (Date, State) rows:", df.duplicated(subset=["Date", "State/UnionTerritory"]).sum())

Names before: 46
Names after:  40
Duplicate (Date, State) rows: 0


The number of names drops from 46 to 40, and renaming created no duplicate rows.

### Problem 3: Hidden missing values in two columns
`isna()` showed no nulls, but `ConfirmedIndianNational` and `ConfirmedForeignNational` use the text `-` as a placeholder for missing data.

In [11]:
df[["ConfirmedIndianNational", "ConfirmedForeignNational"]].value_counts().head()

ConfirmedIndianNational  ConfirmedForeignNational
-                        -                           17664
1                        0                              96
3                        0                              66
2                        0                              29
7                        0                              18
Name: count, dtype: int64

Both columns have `-` in 17,664 of 18,110 rows (97.5%), so they are unusable. Filling them would invent data, and the `Confirmed` column already holds the totals, so both columns are dropped.

In [12]:
df = df.drop(columns=["ConfirmedIndianNational", "ConfirmedForeignNational"])
df.shape

(18110, 5)

### Problem 4: Rows that are not real states
`Unassigned` and `Cases being reassigned to states` are not states or union territories.

In [13]:
fake = ["Unassigned", "Cases being reassigned to states"]
fake_rows = df["State/UnionTerritory"].isin(fake)
print(f"{fake_rows.sum()} rows ({fake_rows.mean():.1%} of the data)")
df[fake_rows].head()

63 rows (0.3% of the data)


,Date,State/UnionTerritory,Cured,Deaths,Confirmed
500,2020-03-30,Unassigned,0,0,46
528,2020-03-31,Unassigned,0,0,38
617,2020-04-03,Unassigned,0,0,77
2133,2020-05-20,Cases being reassigned to states,0,0,1096
2167,2020-05-21,Cases being reassigned to states,0,0,1403


There are 63 such rows in total (3 `Unassigned` and 60 `Cases being reassigned to states`), all from 2020. They are only 0.3% of the data and are not real states, so they are dropped.

In [14]:
df = df[~df["State/UnionTerritory"].isin(fake)]
df.shape

(18047, 5)

### Problem 5: One region under three names
`Dadra and Nagar Haveli`, `Daman & Diu` and `Dadra and Nagar Haveli and Daman and Diu` are the same region. Check the date ranges before merging.

In [15]:
region_names = [
    "Daman & Diu",
    "Dadra and Nagar Haveli",
    "Dadra and Nagar Haveli and Daman and Diu",
]
(df[df["State/UnionTerritory"].isin(region_names)]
   .groupby("State/UnionTerritory")["Date"]
   .agg(["min", "max", "count"]))

,min,max,count
State/UnionTerritory,,,
Dadra and Nagar Haveli,2021-06-04,2021-06-05,2
Dadra and Nagar Haveli and Daman and Diu,2020-05-06,2021-08-11,461
Daman & Diu,2020-06-11,2020-06-11,1


- `Dadra and Nagar Haveli` (2 rows) has no date overlap with the combined name, so it is **renamed** to the combined name.
- `Daman & Diu` (1 row) overlaps with an existing date, so renaming would create a duplicate. It is **dropped**.

In [16]:
combined = "Dadra and Nagar Haveli and Daman and Diu"

df["State/UnionTerritory"] = df["State/UnionTerritory"].replace({"Dadra and Nagar Haveli": combined})
df = df[df["State/UnionTerritory"] != "Daman & Diu"]

df.shape

(18046, 5)

## 4. Final Check and Save

In [17]:
print("Shape:", df.shape)
print("States/UTs:", df["State/UnionTerritory"].nunique())
print("Duplicate (Date, State) rows:", df.duplicated(subset=["Date", "State/UnionTerritory"]).sum())
print("Nulls:", df.isna().sum().sum())

Shape: (18046, 5)
States/UTs: 36
Duplicate (Date, State) rows: 0
Nulls: 0


India has 36 states and union territories, so 36 is the expected number, with no duplicates and no nulls.

In [19]:
df.to_csv("../data/covid_clean.csv", index=False)   # index=False avoids an extra "Unnamed: 0" column
print("Saved to ../data/clean.csv")

Saved to ../data/clean.csv


### Cleaning summary

| # | Problem | Fix |
|---|---|---|
| 1 | `Date` stored as text; `Sno` and `Time` not useful | Converted to datetime; dropped both columns |
| 2 | Typos and star marks in state names (46 names) | Mapped to correct names (40 names) |
| 3 | `-` hiding missing values in 97.5% of two columns | Dropped both columns |
| 4 | 63 rows that are not real states | Dropped |
| 5 | One region under three names | Renamed one name, dropped one row |

**Result:** 18,046 rows, 5 columns (`Date`, `State/UnionTerritory`, `Cured`, `Deaths`, `Confirmed`), 36 states and union territories.